# Phase 1 Data Pre-Processing

## Importing all dependencies and libraries

In [20]:
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
import numpy as np
import pandas as pd
from pydap.client import open_url
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import xarray as xr
import folium
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

print('All imports successful.')


All imports successful.


## Requesting data from the api and storing it locally

In [2]:
import requests

# Check if the dataset os closed or not, of not closed, will throw an access denied error
try:
    ds.close()
except:
    pass

url = (
    "https://ncss.hycom.org/thredds/ncss/GOMb0.01/reanalysis/2024/2d"
    "?var=ssh"
    "&north=31.96&south=18.09&east=-77.0&west=-98.0"
    "&time_start=2024-08-29T00:00:00Z&time_end=2024-09-01T00:00:00Z"
    "&accept=netCDF4"
)

r = requests.get(url, timeout=300, stream=True)
with open('hycom_ssh.nc', 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)

# Open the dataset locally
ds = xr.open_dataset('hycom_ssh.nc')


## Defining the 20+ geolocations for calculating Sea Surface Height(SSH)

In [3]:
LOCATIONS = [
    ('Galveston TX',        29.30, -94.80),
    ('Corpus Christi TX',   27.80, -97.10),
    ('Port Arthur TX',      29.85, -93.90),
    ('South Padre Island',  26.10, -97.20),
    ('New Orleans LA',      29.95, -90.07),
    ('Mississippi Delta',   29.00, -89.25),
    ('Atchafalaya Bay',     29.40, -91.30),
    ('Lake Charles LA',     29.90, -93.25),
    ('Mobile Bay AL',       30.25, -88.10),
    ('Pensacola FL',        30.35, -87.25),
    ('Panama City FL',      30.15, -85.65),
    ('Tampa Bay FL',        27.75, -82.65),
    ('Key West FL',         24.55, -81.80),
    ('Dry Tortugas FL',     24.63, -82.87),
    ('Tampico MX',          22.25, -97.85),
    ('Veracruz MX',         19.20, -96.15),
    ('Campeche MX',         19.85, -90.55),
    ('Progreso MX',         21.30, -89.65),
    ('Cancun MX',           21.15, -86.85),
    ('Deep Water Central',  25.00, -90.00),
    ('Loop Current Core',   26.50, -87.50),
    ('West Florida Shelf',  26.00, -84.50),
    ('Bay of Campeche',     21.00, -93.00),
    ('Yucatan Channel',     22.50, -85.50),
    ('Mississippi River Delta', 29.00, -89.00),
]

print(f'Total locations: {len(LOCATIONS)}')

Total locations: 25


## Extract all the required columns form the dataset

In [5]:
lat_vals  = ds["Latitude"].values     # 1-D array of latitudes
lon_vals  = ds["Longitude"].values     # 1-D array of longitudes
time_vals = ds["MT"].values    # 1-D array of timestamps

print(f"Latitude  shape={lat_vals.shape}  range=[{lat_vals.min():.2f}, {lat_vals.max():.2f}]")
print(f"Longitude  shape={lon_vals.shape}  range=[{lon_vals.min():.2f}, {lon_vals.max():.2f}]")
print(f"Time  shape={time_vals.shape}")


Latitude  shape=(1537,)  range=[18.09, 31.96]
Longitude  shape=(2101,)  range=[-98.00, -77.00]
Time  shape=(73,)


## A function to identify the nearest location to the 20+ specified locations, in case the location is not exactly the same

In [6]:
def nearest_idx(arr, value):
    """Return index of the element in arr closest to value."""
    return int(np.abs(arr - value).argmin())

## Create the final dataset with the nearest latitude and longitude values imputed

In [7]:
rows = []
for name, lat, lon in LOCATIONS:
    lat_idx = nearest_idx(lat_vals, lat)
    lon_idx = nearest_idx(lon_vals, lon)
    for t_idx, t in enumerate(ds['MT'].values):
        rows.append({
            'location':  name,
            'date':      t,
            'latitude':  lat,
            'longitude': lon,
            'ssh':       float(ds['ssh'][t_idx, lat_idx, lon_idx].values),
        })

df = pd.DataFrame(rows)


In [8]:
df.describe()

,date,latitude,longitude,ssh
count,1825,1825.000000,1825.000000,1022.000000
mean,2024-08-30 12:00:00,26.109200,-89.829600,-0.139117
min,2024-08-29 00:00:00,19.200000,-97.850000,-1.041411
25%,2024-08-29 18:00:00,22.500000,-93.250000,-0.342304
50%,2024-08-30 12:00:00,26.500000,-89.650000,-0.173260
75%,2024-08-31 06:00:00,29.400000,-86.850000,0.036861
max,2024-09-01 00:00:00,30.350000,-81.800000,0.579102
std,NaN,3.648047,4.611387,0.284429


### SSH have none values since count of latitude and longitude does not matcch with the count of  ssh, dropping null values

In [9]:
df= df.dropna()

## Plot to show the change in Sea Surface Height(SSH) over the period of 3 days

In [10]:
fig = px.scatter(df, x="date", y="ssh",
                 color="location",
                 title="Plot SSH against Datetime",
                 labels={"date": "Date Time", "ssh": "ssh"})
fig.show()


## Identtify outlier values

In [11]:
valid_min, valid_max = -2.0, 2.0
outliers = df[(df["ssh"] < valid_min) | (df["ssh"] > valid_max)]
print(f"Out-of-range values: {len(outliers)}")
print(outliers)



Out-of-range values: 0
Empty DataFrame
Columns: [location, date, latitude, longitude, ssh]
Index: []


## Display all location on the Map

In [12]:
loc_df = pd.DataFrame(LOCATIONS, columns=['name','latitude','longitude'])
fig = px.scatter_map(loc_df, lat="latitude", lon="longitude", hover_name="name", zoom=3)
fig.show()

# Phase 2: Training and Evaluating Model

## Sliding Window Function

In [13]:

WINDOW     = 8   # 8 hours window
HORIZON    = 1   # h=1

def build_sliding_window(df, window=8, horizon=1):
    """This function transforms a time-series DataFrame into a supervised machine 
    learning dataset using a sliding window approach."""
    df["hour"] = df["date"].dt.hour

    feature_cols = ["ssh", "latitude", "longitude", "hour"]
    data = df[feature_cols].values

    X, y = [], []
    for i in range(window, len(df) - horizon + 1):
        window_block = data[i - window : i].flatten()   
        target       = data[i + horizon - 1, 0]        
        X.append(window_block)
        y.append(target)

    feature_names = [
        f"{col}_t-{window - j}"
        for j in range(window)
        for col in feature_cols
    ]
    return np.array(X), np.array(y), feature_names

X, y, feature_names = build_sliding_window(df, WINDOW, HORIZON)
print(f"\nSliding window applied → X: {X.shape}, y: {y.shape}")




Sliding window applied → X: (1014, 32), y: (1014,)


## Train Test Split

In [14]:
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")



Train size: 811 | Test size: 203


## Scaling values

In [15]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)



## Training Linear Regression Basemodel

In [16]:
model = LinearRegression()
model.fit(X_train, y_train)



,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


## Evaluation of Basemodel

In [17]:

y_pred = model.predict(X_test)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("\n" + "─" * 50)
print("         MODEL EVALUATION")
print("─" * 50)
print(f"  MSE  : {mse:.6f}")
print(f"  RMSE : {rmse:.6f}")
print("─" * 50)


──────────────────────────────────────────────────
         MODEL EVALUATION
──────────────────────────────────────────────────
  MSE  : 0.009873
  RMSE : 0.099365
──────────────────────────────────────────────────


## Sample Prediction

In [18]:
results = pd.DataFrame({
    "Actual SSH" : y_test[:10],
    "Predicted SSH": y_pred[:10],
    "Error"      : np.abs(y_test[:10] - y_pred[:10])
})
print("\nSample Predictions (first 10):")
print(results.to_string(index=False))


Sample Predictions (first 10):
 Actual SSH  Predicted SSH    Error
  -0.202641      -0.178020 0.024622
  -0.275343      -0.243959 0.031383
  -0.360049      -0.326078 0.033971
  -0.450429      -0.417934 0.032495
  -0.535620      -0.510059 0.025561
  -0.602262      -0.588947 0.013314
  -0.639786      -0.640375 0.000589
  -0.638459      -0.656426 0.017967
  -0.595097      -0.629891 0.034794
  -0.513804      -0.558282 0.044478


## calculating the linear regresson mean square error, root mean squared error and r2 error for comparison

In [21]:
lr_mse  = mean_squared_error(y_test, y_pred)
lr_rmse = np.sqrt(lr_mse)
lr_r2   = r2_score(y_test, y_pred)

## Gradient Boosting Regressor Model

In [22]:
gb = GradientBoostingRegressor(
    n_estimators   = 300,
    learning_rate  = 0.06, # most optimal learning rate, tested with 0.04, 0.05, 0.06, 0.07, 0.08
    max_depth      = 4,
    subsample      = 0.8,
    min_samples_leaf= 5,
    random_state   = 42
)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
 
gb_mse  = mean_squared_error(y_test, gb_pred)
gb_rmse = np.sqrt(gb_mse)
gb_r2   = r2_score(y_test, gb_pred)

## Printing the final evaluation with comparison of basemodel and boosting model

In [23]:

mse_imp  = ((lr_mse  - gb_mse)  / lr_mse)  * 100
rmse_imp = ((lr_rmse - gb_rmse) / lr_rmse) * 100
r2_imp   = gb_r2 - lr_r2
 
print("=" * 54)
print("           MODEL EVALUATION COMPARISON")
print("=" * 54)
print(f"{'Metric':<18} {'Linear Reg':>12} {'Grad Boost':>12}")
print("-" * 54)
print(f"{'MSE':<18} {lr_mse:>12.6f} {gb_mse:>12.6f}")
print(f"{'RMSE':<18} {lr_rmse:>12.6f} {gb_rmse:>12.6f}")
print(f"{'R² Score':<18} {lr_r2:>12.6f} {gb_r2:>12.6f}")
print("=" * 54)
print(f"\n  MSE  improvement  : {mse_imp:.2f}%")
print(f"  RMSE improvement  : {rmse_imp:.2f}%")
print(f"  R²   improvement  : +{r2_imp:.4f}")
print("=" * 54)
 


           MODEL EVALUATION COMPARISON
Metric               Linear Reg   Grad Boost
------------------------------------------------------
MSE                    0.009873     0.007512
RMSE                   0.099365     0.086674
R² Score               0.913696     0.934334

  MSE  improvement  : 23.91%
  RMSE improvement  : 12.77%
  R²   improvement  : +0.0206
